<h3 align='center'>AIRLINE</h3>
<h3 align='center'>AIRLINE OPERATIONS, DELAY & RELIABILITY ANALYTICS</h3>

<h4 align='center'>NOTEBOOK 04 - DATA QUALITY</h4>

**Objective:** Assess the quality, completeness, consistency and validity
of the raw BTS airline operational dataset BEFORE cleaning.

**IMPORTANT:** Raw data will NOT be modified in this notebook.

Assessment Areas:

1. Dataset structure
2. Missingness
3. Duplicate records
4. Data types
5. Unique-value consistency
6. Range / domain checks
7. Business-rule checks
8. Data-quality summary
9. Cleaning recommendations

## Import Paths

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Project Paths

In [2]:
PROJECT_PATH = Path.cwd().parent

RAW_PATH = PROJECT_PATH / "01_Raw_Data"
DOCUMENTATION_PATH = PROJECT_PATH / "02_Documentation"
RAW_2024_PATH = RAW_PATH / "2024"
EXTRACT_PATH = RAW_2024_PATH / "January"

print("Project Path:", PROJECT_PATH.resolve())
print("Raw Data Path:", RAW_PATH.resolve())
print("Documentation Path:", DOCUMENTATION_PATH.resolve())
print("RAW_2024_PATH:", RAW_2024_PATH)
print("EXTRACT_PATH:", EXTRACT_PATH)

Project Path: D:\arc\Python\Python Projects\airline_operations_analytics
Raw Data Path: D:\arc\Python\Python Projects\airline_operations_analytics\01_Raw_Data
Documentation Path: D:\arc\Python\Python Projects\airline_operations_analytics\02_Documentation
RAW_2024_PATH: d:\arc\Python\Python Projects\airline_operations_analytics\01_Raw_Data\2024
EXTRACT_PATH: d:\arc\Python\Python Projects\airline_operations_analytics\01_Raw_Data\2024\January


## Load Raw Data

In [3]:
csv_files = list(
    (RAW_PATH / "2024").rglob("*.csv")
    )

if not csv_files:
    raise FileNotFoundError( "No CSV file found inside 01_Raw_Data/2024")

RAW_FILE = csv_files[0]

df = pd.read_csv(RAW_FILE, low_memory= False)

print("File:", RAW_FILE.name)
print("Shape:", df.shape)

File: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv
Shape: (547271, 110)


## Data Quality Assessment

In [4]:
baseline = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Total Cells",
        "Duplicate Rows",
        "Completely Empty Columns",
        "Columns With Missing Values",
        "Columns Without Missing Values"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df. shape[0] * df. shape[1],
        df.duplicated().sum(),
        df.isna().all().sum(),
        (df.isna().any()).sum(),
        (~df.isna().any()).sum()
    ]
})

display(baseline)

,Metric,Value
0,Total Rows,547271
1,Total Columns,110
2,Total Cells,60199810
3,Duplicate Rows,0
4,Completely Empty Columns,17
5,Columns With Missing Values,71
6,Columns Without Missing Values,39


## Structural Assessment

In [5]:
print("========== DATASET STRUCTURE ==========")

print("Rows:", df.shape[0])
print("Columns:", df. shape[1])

print("\nMemory Usage:")
print(
    f"{df.memory_usage(deep=True).sum() / (1024 ** 2) :.2f} MB"
)

========== DATASET STRUCTURE ==========
Rows: 547271
Columns: 110

Memory Usage:
894.03 MB


## Column-Level Data Inventory

In [6]:
data_inventory = pd. DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str),
    "Non_Null_Count": df.notna().sum().values,
    "Null_Count": df.isna().sum().values,
    "Null_Percentage": (
        df.isna().mean().values * 100
    ).round(2),
    "Unique_Values": df.nunique(
        dropna=True
    ).values

})

display(data_inventory)

,Column,Data_Type,Non_Null_Count,Null_Count,Null_Percentage,Unique_Values
Year,Year,int64,547271,0,0.00,1
Quarter,Quarter,int64,547271,0,0.00,1
Month,Month,int64,547271,0,0.00,1
DayofMonth,DayofMonth,int64,547271,0,0.00,31
DayOfWeek,DayOfWeek,int64,547271,0,0.00,7
...,...,...,...,...,...,...
Div5TotalGTime,Div5TotalGTime,float64,0,547271,100.00,0
Div5LongestGTime,Div5LongestGTime,float64,0,547271,100.00,0
Div5WheelsOff,Div5WheelsOff,float64,0,547271,100.00,0
Div5TailNum,Div5TailNum,float64,0,547271,100.00,0


## Missingness Profile

In [7]:
missing_profile = (
    data_inventory[
        data_inventory["Null_Count"] > 0
    ]
    .sort_values(
        "Null_Percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

display(missing_profile)

,Column,Data_Type,Non_Null_Count,Null_Count,Null_Percentage,Unique_Values
0,Div5WheelsOff,float64,0,547271,100.00,0
1,Unnamed: 109,float64,0,547271,100.00,0
2,Div4LongestGTime,float64,0,547271,100.00,0
3,Div4TotalGTime,float64,0,547271,100.00,0
4,Div4WheelsOn,float64,0,547271,100.00,0
5,Div4AirportSeqID,float64,0,547271,100.00,0
6,Div4AirportID,float64,0,547271,100.00,0
7,Div3AirportID,float64,1,547270,100.00,1
8,Div3AirportSeqID,float64,1,547270,100.00,1
9,Div3WheelsOn,float64,1,547270,100.00,1


## Missingness Classification

In [8]:
missing_profile["Missingness_Category"] = np.select(
    [
        missing_profile["Null_Percentage"] == 0,
        missing_profile["Null_Percentage"] < 5,
        missing_profile["Null_Percentage"] < 30,
        missing_profile["Null_Percentage"] < 70,
        missing_profile["Null_Percentage"] < 100,
        missing_profile["Null_Percentage"] == 100
    ],
    [
        "Complete",
        "Low",
        "Moderate",
        "High",
        "Very High",
        "Completely Missing"
    ],
    default="Review"
)

display(missing_profile)

,Column,Data_Type,Non_Null_Count,Null_Count,Null_Percentage,Unique_Values,Missingness_Category
0,Div5WheelsOff,float64,0,547271,100.00,0,Completely Missing
1,Unnamed: 109,float64,0,547271,100.00,0,Completely Missing
2,Div4LongestGTime,float64,0,547271,100.00,0,Completely Missing
3,Div4TotalGTime,float64,0,547271,100.00,0,Completely Missing
4,Div4WheelsOn,float64,0,547271,100.00,0,Completely Missing
5,Div4AirportSeqID,float64,0,547271,100.00,0,Completely Missing
6,Div4AirportID,float64,0,547271,100.00,0,Completely Missing
7,Div3AirportID,float64,1,547270,100.00,1,Completely Missing
8,Div3AirportSeqID,float64,1,547270,100.00,1,Completely Missing
9,Div3WheelsOn,float64,1,547270,100.00,1,Completely Missing


## Completely Empty Columns

In [9]:
empty_columns = df.columns [
    df.isna().all()
].tolist()

print("Completely Empty Columns:", len(empty_columns))

for column in empty_columns:
    print(column)

Completely Empty Columns: 17
Div4Airport
Div4AirportID
Div4AirportSeqID
Div4WheelsOn
Div4TotalGTime
Div4LongestGTime
Div4WheelsOff
Div4TailNum
Div5Airport
Div5AirportID
Div5AirportSeqID
Div5WheelsOn
Div5TotalGTime
Div5LongestGTime
Div5WheelsOff
Div5TailNum
Unnamed: 109


## Duplicate Assessment

In [10]:
duplicate_count = df.duplicated().sum()

print("Complete Duplicate Rows:", duplicate_count)

Complete Duplicate Rows: 0


## Data-Type Quality Assessment

In [11]:
object_columns = df.select_dtypes(
    include="object"
).columns.tolist()

print("object Columns:", len(object_columns))

for column in object_columns:
    print(column)

object Columns: 21
FlightDate
Reporting_Airline
IATA_CODE_Reporting_Airline
Tail_Number
Origin
OriginCityName
OriginState
OriginStateName
Dest
DestCityName
DestState
DestStateName
DepTimeBlk
ArrTimeBlk
CancellationCode
Div1Airport
Div1TailNum
Div2Airport
Div2TailNum
Div3Airport
Div3TailNum


C:\Users\asus\AppData\Local\Temp\ipykernel_9272\4094765401.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df.select_dtypes(


## Date Quality

In [12]:
flight_date_test = pd.to_datetime(
    df["FlightDate"],
    errors="coerce"
)

invalid_flight_dates = (
    flight_date_test.isna() &
    df["FlightDate"].notna()
).sum()

print(
    "Invalid / Unparseable FlightDate:",
    invalid_flight_dates
)

print(
    "Minimum Flight Date:",
    flight_date_test.min()
)

print(
    "Maximum Flight Date:",
    flight_date_test.max()
)

Invalid / Unparseable FlightDate: 0
Minimum Flight Date: 2024-01-01 00:00:00
Maximum Flight Date: 2024-01-31 00:00:00


## Categorical Consistency

In [13]:
categorical_columns = [
    "Reporting Airline",
    "Origin",
    "Dest",
    "CancellationCode",
    "DepTimeBlk",
    "ArrTimeBlk"
]

categorical_columns = [
    c for c in categorical_columns
    if c in df.columns
]

for column in categorical_columns:
    print(f"\n========== {column} ==========")
    print(df[column].value_counts(dropna=False).head(20))


========== Origin ==========
Origin
ATL    26315
DFW    23570
DEN    23361
ORD    20321
CLT    16378
PHX    15378
LAX    15228
LAS    14942
MCO    14296
LGA    12700
SEA    11751
DCA    11486
MIA    10214
EWR    10176
BOS    10134
SFO    10133
DTW     9473
JFK     9471
SLC     9459
MSP     9438
Name: count, dtype: int64

========== Dest ==========
Dest
ATL    26294
DFW    23588
DEN    23357
ORD    20327
CLT    16382
PHX    15367
LAX    15220
LAS    14943
MCO    14304
LGA    12718
SEA    11757
DCA    11487
MIA    10210
EWR    10179
SFO    10128
BOS    10126
DTW     9470
JFK     9467
SLC     9455
MSP     9431
Name: count, dtype: int64

========== CancellationCode ==========
CancellationCode
NaN    526882
B       12085
A        7736
C         568
Name: count, dtype: int64

========== DepTimeBlk ==========
DepTimeBlk
0600-0659    39849
0700-0759    37556
0800-0859    36756
1000-1059    35162
1800-1859    34692
1700-1759    34437
1300-1359    34095
1100-1159    34041
1200-1259    33282
140

## Binary Field Assessment

In [14]:
binary_columns = [
    "Cancelled",
    "Diverted",
    "DepDel15",
    "ArrDel15"
]

for column in binary_columns:

    if column in df.columns:

        print(f"\n========== {column} ==========")

        print(
            df[column].value_counts(dropna=False)
        )


========== Cancelled ==========
Cancelled
0.00    526882
1.00     20389
Name: count, dtype: int64

========== Diverted ==========
Diverted
0.00    545759
1.00      1512
Name: count, dtype: int64

========== DepDel15 ==========
DepDel15
0.00    405154
1.00    122259
NaN      19858
Name: count, dtype: int64

========== ArrDel15 ==========
ArrDel15
0.00    398960
1.00    126410
NaN      21901
Name: count, dtype: int64


In [18]:
range_checks = {
    "Quarter outside 1-4":
        (~df["Quarter"].between(1, 4)).sum(),

    "Month outside 1-12":
        (~df["Month"].between(1, 12)).sum(),

    "DayOfWeek outside 1-7":
        (~df["DayOfWeek"].between(1, 7)).sum(),

    "Distance <= 0":
        (df["Distance"] <= 0).sum(),

    "Flights <= 0":
        (df["Flights"] <= 0).sum()
}

range_quality = pd.DataFrame({
    "Validation_Rule": range_checks.keys(),
    "Violation_Count": range_checks.values()
})

display(range_quality)

,Validation_Rule,Violation_Count
0,Quarter outside 1-4,0
1,Month outside 1-12,0
2,DayOfWeek outside 1-7,0
3,Distance <= 0,0
4,Flights <= 0,0


In [13]:
data_validity = df[["Cancelled", "Diverted", "DepDelay"]].sum()

data_validity

Cancelled      20389.0
Diverted        1512.0
DepDelay     8280743.0
dtype: float64

In [14]:
quality_assessment = pd.DataFrame ({
    "Column_Name": df.columns,
    "Data_Types": df.dtypes.values,
    "Missing_Count": df.isna().sum().values
})

quality_assessment

,Column_Name,Data_Types,Missing_Count
0,Year,int64,0
1,Quarter,int64,0
2,Month,int64,0
3,DayofMonth,int64,0
4,DayOfWeek,int64,0
...,...,...,...
105,Div5TotalGTime,float64,547271
106,Div5LongestGTime,float64,547271
107,Div5WheelsOff,float64,547271
108,Div5TailNum,float64,547271


In [15]:
quality_assessment.to_csv(
    DOCUMENTATION_PATH / "data_quality_assessment.csv",
        index = False
)

print("Quality Assessment Saved Successfully")

Quality Assessment Saved Successfully
